# Project: Tomographic reconstruction with Huber regularization (3D reconstruction)

The noise in the 3D data has a relative norm of about 1/100, so we use $\delta = 0.01 \|y\|$.
The parameters $\epsilon$ and $\tau$ are taken from the 2D tests (`project_2d.ipynb`).
The functions shared with the 2D notebook are in `utils.py`.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import odl
from odl.applications import tomo
from odl.contrib.tomo import elekta_icon_geometry

from utils import save_figure, show_images, show_slices, axial_slice, huber_reconstruction, parker_weights

In [ ]:
eps = 1e-3  # chosen in the 2D tests
tau = 1.1
scalings = [1.0, 0.5, 0.25]

In [ ]:
reco_space = odl.uniform_discr([-112, -112, 0], [112, 112, 224], (128, 128, 128), dtype='float32')
geometry = elekta_icon_geometry(num_angles=128)
ray_trafo = tomo.RayTransform(vol_space=reco_space, geometry=geometry)
data_space = ray_trafo.range

data_path = '/home/jupyter-miniproject8/InverseProblems-Labs/noisy_sinogram.npy'
data = data_space.element(np.load(data_path).astype('float32', copy=False))

delta_3d = 0.01 * data.norm()
print(f'Data shape: {data_space.shape}, delta = {delta_3d:.2f}')

## Filtered back-projection in 3D

In [ ]:
fbp = tomo.fbp_op(ray_trafo)
show_slices(fbp(data), 'FBP without Parker weighting', filename='fbp_3d_no_parker')

# the weighted data is computed once and reused for all the FBPs, to save memory
weighted_data = parker_weights(ray_trafo) * data
x_fbp_3d = fbp(weighted_data)
show_slices(x_fbp_3d, 'FBP with Parker weighting', filename='fbp_3d_parker')
print(f'min = {odl.min(x_fbp_3d):.3f}, max = {odl.max(x_fbp_3d):.3f}')

In [ ]:
# display window centred on the brain values (central block of the FBP volume)
brain = x_fbp_3d.asarray()[48:80, 48:80, 60:68]
print(f'brain: mean = {brain.mean():.3f}, std = {brain.std():.3f}')
clim_3d = (brain.mean() - 3 * brain.std(), brain.mean() + 3 * brain.std())

In [ ]:
for filter_type in ['Ram-Lak', 'Hann']:
    images = []
    for scaling in scalings:
        fbp = tomo.fbp_op(ray_trafo, filter_type=filter_type, frequency_scaling=scaling)
        images.append(axial_slice(fbp(weighted_data)))
    show_images(images, [f'{filter_type}, scaling = {scaling}' for scaling in scalings], clim=clim_3d,
                filename=f'fbp_3d_{filter_type}')

x_hann_3d = tomo.fbp_op(ray_trafo, filter_type='Hann', frequency_scaling=0.25)(weighted_data)

# the weighted data is not needed anymore, this frees memory for the Huber reconstructions
del weighted_data

## Huber reconstruction in 3D

In [ ]:
t = time.time()
ray_trafo_norm = float(ray_trafo.norm(estimate=True))
grad_norm = float(odl.Gradient(reco_space).norm(estimate=True))
op_norms_3d = (ray_trafo_norm, grad_norm)
print(f'||A|| = {ray_trafo_norm:.2f}, ||grad|| = {grad_norm:.2f} ({time.time() - t:.0f} s)')

### Short test to measure the time per iteration

In [ ]:
t = time.time()
x, _ = huber_reconstruction(ray_trafo, data, lam=1.0, eps=eps, niter=10, op_norms=op_norms_3d)
print(f'{(time.time() - t) / 10:.2f} s per iteration')
print(f'residual / delta = {(ray_trafo(x) - data).norm() / delta_3d:.3f}')
show_slices(x, r'10 iterations, $\lambda = 1$', clim=clim_3d, filename='huber_3d_test')

### Choice of $\lambda$ with Morozov's discrepancy principle

With $\delta = 0.01 \|y\|$ the residual never gets below about $1.34\delta$, even with almost no regularization,
so the data contains errors that the model cannot explain. We therefore estimate the total data error $\delta_{\text{eff}}$
as the residual of the almost unregularized reconstruction, and pick the first $\lambda$ with $\| A x - y \| \le \tau \delta_{\text{eff}}$.
Each $\lambda$ starts from the reconstruction of the previous one to save iterations.

In [ ]:
niter_3d = 200
lam_values_3d = [1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001]

recos_3d = []
ratios_3d = []
x = None
for lam in lam_values_3d:
    t = time.time()
    x, _ = huber_reconstruction(ray_trafo, data, lam, eps, niter_3d, op_norms_3d, x0=x)
    recos_3d.append(x)
    ratios_3d.append((ray_trafo(x) - data).norm() / delta_3d)
    print(f'lambda = {lam}: residual / delta = {ratios_3d[-1]:.3f} ({time.time() - t:.0f} s)')

# residual of the almost unregularized reconstruction
delta_eff = min(ratios_3d) * delta_3d
print(f'delta_eff = {delta_eff / delta_3d:.3f} delta')

# first (largest) lambda that satisfies the discrepancy principle with delta_eff
for i_morozov_3d, ratio in enumerate(ratios_3d):
    if ratio * delta_3d <= tau * delta_eff:
        break
lam_morozov_3d = lam_values_3d[i_morozov_3d]
print(f'Morozov: lambda = {lam_morozov_3d}')

In [ ]:
plt.figure()
plt.semilogx(lam_values_3d, ratios_3d, 'o-', label='residual')
plt.axhline(tau, color='r', linestyle='--', label=r'$\tau \delta$')
plt.axhline(tau * delta_eff / delta_3d, color='g', linestyle='--', label=r'$\tau \delta_{\mathrm{eff}}$')
plt.axvline(lam_morozov_3d, color='k', linestyle=':', label=r'Morozov $\lambda$')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$\|Ax - y\| / \delta$')
plt.grid(True)
plt.legend()
save_figure('morozov_3d')
plt.show()

show_images([axial_slice(x) for x in recos_3d], [rf'$\lambda = {lam}$' for lam in lam_values_3d], clim=clim_3d,
            filename='lambda_3d')

### Check of the largest $\lambda$

$\lambda = 1$ is the first value of the sweep, so it starts from zero and has the smallest step size.
We continue it to see if its residual goes below $\tau \delta_{\text{eff}}$ once it has converged.

In [ ]:
x_check = recos_3d[0]
for k in range(2, 6):
    x_check, _ = huber_reconstruction(ray_trafo, data, lam_values_3d[0], eps, 200, op_norms_3d, x0=x_check)
    ratio = (ray_trafo(x_check) - data).norm() / delta_3d
    print(f'{200 * k} iterations: residual / delta = {ratio:.3f} (threshold = {tau * delta_eff / delta_3d:.3f})')

show_images([axial_slice(recos_3d[0]), axial_slice(x_check)],
            [rf'$\lambda = {lam_values_3d[0]}$, 200 iterations', rf'$\lambda = {lam_values_3d[0]}$, 1000 iterations'],
            clim=clim_3d, filename='lambda_check_3d')

### Evolution with the number of iterations

In [ ]:
checkpoints = [10, 50, 200, 500, 1000]

# steepest descent with a constant step has no memory, so we can continue from the last iterate
iterates = []
x = None
done = 0
for k in checkpoints:
    x, _ = huber_reconstruction(ray_trafo, data, lam_morozov_3d, eps, k - done, op_norms_3d, x0=x)
    done = k
    iterates.append(x)
    print(f'{k} iterations: residual / delta = {(ray_trafo(x) - data).norm() / delta_3d:.3f}')

show_images([axial_slice(x) for x in iterates], [f'{k} iterations' for k in checkpoints], clim=clim_3d,
            filename='iterations_3d')

In [ ]:
x_huber_3d = iterates[-1]

show_slices(x_fbp_3d, 'FBP (Ram-Lak, scaling = 1.0)', clim=clim_3d, filename='final_fbp_ramlak')
show_slices(x_hann_3d, 'FBP (Hann, scaling = 0.25)', clim=clim_3d, filename='final_fbp_hann')
show_slices(x_huber_3d, rf'Huber, $\lambda = {lam_morozov_3d}$, {checkpoints[-1]} iterations', clim=clim_3d,
            filename='final_huber')